# Ad-Hoc Exploratory Analysis

### Setup & Imports

In [7]:
import duckdb
import pandas as pd
import plotly.express as px

# from IPython.core.display import HTML
# def css_styling():
#     styles = open("./custom.css", "r").read()
#     return HTML(f"<style>{styles}</style>")
# css_styling()

# Connect to in-memory DuckDB instance
conn = duckdb.connect()

print("DuckDB initialized successfully!")

DuckDB initialized successfully!


### Querying Parquet Files Directly with SQL
#### Query fact and dimension tables directly from Parquet files without loading into a database first

In [8]:

query = """
SELECT 
    pr.repo_name,
    pr.pull_request_id,
    pr.source_branch,
    pr.pr_state,
    a.author_username,
    pr.created_at,
    pr.merged_at,
    round(EPOCH(pr.merged_at - pr.created_at) / 3600.0 ,2) AS lead_time_hours
FROM '../data/exports/fct_pull_requests.parquet' pr
LEFT JOIN '../data/exports/dim_authors.parquet' a
    ON pr.author_username = a.author_username
WHERE pr.is_merged
"""

df_merged_prs = conn.execute(query).df()
df_merged_prs.head()

,repo_name,pull_request_id,source_branch,pr_state,author_username,created_at,merged_at,lead_time_hours
0,duckdb,4143259897,fix-vector-operations-warning,closed,smvv,2026-07-27 14:26:14+00:00,2026-07-27 18:09:23+00:00,3.72
1,duckdb,4142367421,fix-main-jul-27,closed,smvv,2026-07-27 12:35:20+00:00,2026-07-27 15:18:10+00:00,2.71
2,duckdb,4140510472,hjiang/fix-null-check,closed,dentiny,2026-07-27 08:04:15+00:00,2026-07-27 11:53:33+00:00,3.82
3,duckdb,4137815575,fix-versioning-tag-quoting,closed,jokasimr,2026-07-26 21:14:05+00:00,2026-07-27 18:23:28+00:00,21.16
4,duckdb,4137319418,feature/parquet-bss-int,closed,hertelukas,2026-07-26 18:22:20+00:00,2026-07-27 18:40:09+00:00,24.30


### Ad-Hoc Metric — Lead Time Distribution by Repository
#### Distribution of Lead Time to Merge across repositories

In [9]:
fig = px.box(
    df_merged_prs, 
    x="repo_name", 
    y="lead_time_hours", 
    points="all",
    title="Ad-Hoc Analysis: Lead Time to Merge Distribution (Hours)",
    labels={"lead_time_hours": "Lead Time (Hours)", "repository": "Repository"}
)
fig.show()

### Top Contributors Analysis
#### Aggregate PR throughput and average lead time per author

In [10]:
author_summary = conn.execute("""
    SELECT 
        a.author_username,
        COUNT(pr.pull_request_id) AS total_merged_prs,
        ROUND(AVG(EPOCH(pr.merged_at - pr.created_at) / 3600.0), 2) AS avg_lead_time_hours,
        ROUND(MEDIAN(EPOCH(pr.merged_at - pr.created_at) / 3600.0), 2) AS median_lead_time_hours
    FROM '../data/exports/fct_pull_requests.parquet' pr
    JOIN '../data/exports/dim_authors.parquet' a ON pr.author_username = a.author_username
    WHERE pr.is_merged
    GROUP BY a.author_username
    ORDER BY total_merged_prs DESC
    LIMIT 10
""").df()

author_summary

,author_username,total_merged_prs,avg_lead_time_hours,median_lead_time_hours
0,renovate[bot],323,47.86,27.22
1,github-actions[bot],167,8.38,1.01
2,dependabot[bot],158,14.22,2.60
3,lukasmasuch,90,76.82,30.02
4,dentiny,74,47.05,27.13
5,charliermarsh,73,46.14,18.67
6,Mytherin,47,13.58,3.95
7,ntBre,46,50.18,20.70
8,carljm,36,14.38,3.47
9,carlopi,35,32.38,17.39


### Repository Metrics
#### Open time and percentage of merges

In [11]:
df = duckdb.query("""
    SELECT repo_name, AVG(total_open_hours) as avg_open_time,  round(sum(is_merged)/count(*)*100,1) as avg_merged
    FROM '../data/exports/fct_pull_requests.parquet'
    GROUP BY repo_name
    """).df()

In [12]:
import plotly.graph_objects as go

repo_names = df["repo_name"].tolist()
avg_open_time = df["avg_open_time"].tolist()
avg_merged = df["avg_merged"].tolist()

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=repo_names,
        y=avg_open_time,
        name="Avg Open Time",
        marker_color="#636efa",
        yaxis="y",
        offsetgroup="open",
        width=0.35,
    )
)

fig.add_trace(
    go.Bar(
        x=repo_names,
        y=avg_merged,
        name="Avg Merged (%)",
        marker_color="#EF553B",
        yaxis="y2",
        offsetgroup="merged",
        width=0.35,
    )
)

fig.update_layout(
    title="Repository Metrics Overview",
    barmode="group",
    xaxis_title="Repository",
    yaxis=dict(title="Avg Open Time (Hours)"),
    yaxis2=dict(title="Avg Merged (%)", overlaying="y", side="right"),
    legend_title_text="Metric",
)

fig.show()